# Task 5: Model training under leave-one-province-out CV
Rules enforced in every fold:
- Preprocessing is re-fit on fold-train rows only (nested, no leakage)
- Augmentation is generated from fold-train rows only
- Held-out province rows are real, untouched, and used only for scoring
- All 4 models see the identical synthetic rows within a fold (fair comparison)
Models: Linear (reported baseline), Ridge, Random Forest, constrained LightGBM.

In [5]:
# setup, data, and model roaster
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd, joblib

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.utils.augmentation import stratified_mixup

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv(PROJECT_ROOT / "data/processed/merged_rice_yield_dataset.csv")
df["temp_mean_c"] = (df["temp_max_c"] + df["temp_min_c"]) / 2
df["temp_range_c"] = df["temp_max_c"] - df["temp_min_c"]

split_info = json.load(open(PROJECT_ROOT / "artifacts/split_provinces.json"))
df_train = df[df["province"].isin(split_info["train_provinces"])].copy()

X_train_raw = df_train.drop(columns=["yield_mt_ha", "province"])
y_train = df_train["yield_mt_ha"].to_numpy()
strata = (df_train["season_period"] + "|" + df_train["ecosystem"]
          + "|" + df_train["island_group"]).to_numpy()
province_of_row = df_train["province"].to_numpy()
provinces = sorted(df_train["province"].unique())

CAT_COLS = ["season_period", "ecosystem", "island_group"]
NUM_STD = ["n_application_rate", "p_application_rate", "k_application_rate",
           "total_fertilizer_kg_ha", "fertilizer_diversification_index",
           "rainfall_mm", "solar_radiation_mj_m2", "soil_ph",
           "temp_mean_c", "temp_range_c"]
NUM_LOG = ["area_harvested_ha"]

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("num_std", Pipeline([("scaler", StandardScaler())]), NUM_STD),
        ("num_log", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                              ("scaler", StandardScaler())]), NUM_LOG),
        ("cat", Pipeline([("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]), CAT_COLS),
    ])

def make_models():
    return {
        "linear_baseline": LinearRegression(),
        "ridge": Ridge(alpha=1.0, random_state=42),
        "random_forest": RandomForestRegressor(
            n_estimators=500, min_samples_leaf=2, random_state=42, n_jobs=-1),
        "lightgbm": LGBMRegressor(
            n_estimators=200, learning_rate=0.05, num_leaves=7, max_depth=3,
            min_child_samples=25, colsample_bytree=0.8,
            reg_alpha=1.0, reg_lambda=5.0, random_state=42, verbose=-1),
    }

print(f"Training provinces: {len(provinces)} | rows: {len(df_train)}")

Training provinces: 60 | rows: 229


In [6]:
# LOPO cv engine
def lopo_cv(multiplier, province_list, base_seed=42):
    model_names = list(make_models().keys())
    oof = {name: np.full(len(df_train), np.nan) for name in model_names}
    fold_rmse = {name: [] for name in model_names}
    fold_mae = {name: [] for name in model_names}

    for f, prov in enumerate(province_list):
        tr = province_of_row != prov
        va = province_of_row == prov

        prep = make_preprocessor()
        Xf = prep.fit_transform(X_train_raw[tr])
        Xv = prep.transform(X_train_raw[va])
        if hasattr(Xf, "toarray"):
            Xf, Xv = Xf.toarray(), Xv.toarray()
        yf, yv, sf = y_train[tr], y_train[va], strata[tr]

        Xa, ya, wa = stratified_mixup(Xf, yf, sf, multiplier=multiplier,
                                      random_state=base_seed + f)

        for name, model in make_models().items():
            model.fit(Xa, ya, sample_weight=wa)
            pred = model.predict(Xv)
            oof[name][va] = pred
            fold_rmse[name].append(np.sqrt(mean_squared_error(yv, pred)))
            fold_mae[name].append(mean_absolute_error(yv, pred))

    return oof, fold_rmse, fold_mae

In [7]:
# metrics summary
def summarize(oof, fold_rmse, fold_mae, multiplier):
    rows = []
    for name in oof:
        mask = ~np.isnan(oof[name])
        yt, yp = y_train[mask], oof[name][mask]
        rows.append({
            "model": name, "multiplier": multiplier,
            "oof_rmse": np.sqrt(mean_squared_error(yt, yp)),
            "oof_mae": mean_absolute_error(yt, yp),
            "oof_r2": r2_score(yt, yp),
            "fold_rmse_mean": np.mean(fold_rmse[name]),
            "fold_rmse_std": np.std(fold_rmse[name]),
            "fold_mae_mean": np.mean(fold_mae[name]),
            "fold_mae_std": np.std(fold_mae[name]),
            "n_folds": len(fold_rmse[name]),
        })
    return pd.DataFrame(rows)

In [8]:
# full sweep configuration
import time

SMOKE = False
mults = [1, 2, 3, 5]
prov_list = provinces

results, oof_store = [], {}
t0 = time.time()

for m in mults:
    print(f"Running multiplier {m}x...")
    oof, frmse, fmae = lopo_cv(m, prov_list)
    results.append(summarize(oof, frmse, fmae, m))
    
    for name in oof:
        oof_store[f"{name}_m{m}"] = oof[name]
        
print(f"\nTotal Elapsed: {time.time() - t0:.1f}s")

metrics_df = pd.concat(results)
print("\n--- Full LOPO CV Results ---")
print(metrics_df.round(4).to_string(index=False))

metrics_df.to_csv(PROJECT_ROOT / "artifacts/model_metrics.csv", index=False)

oof_df = pd.DataFrame(oof_store)
oof_df.to_csv(PROJECT_ROOT / "artifacts/oof_predictions.csv", index=False)

print("\nSaved: artifacts/model_metrics.csv")
print("Saved: artifacts/oof_predictions.csv")

Running multiplier 1x...
Running multiplier 2x...
Running multiplier 3x...
Running multiplier 5x...

Total Elapsed: 147.9s

--- Full LOPO CV Results ---
          model  multiplier  oof_rmse  oof_mae  oof_r2  fold_rmse_mean  fold_rmse_std  fold_mae_mean  fold_mae_std  n_folds
linear_baseline           1    0.6772   0.5511  0.3673          0.6115         0.2930         0.5527        0.2820       60
          ridge           1    0.6749   0.5488  0.3716          0.6094         0.2920         0.5503        0.2814       60
  random_forest           1    0.6758   0.5334  0.3700          0.6068         0.3006         0.5351        0.2887       60
       lightgbm           1    0.7027   0.5589  0.3188          0.6319         0.3150         0.5622        0.3005       60
linear_baseline           2    0.6816   0.5552  0.3591          0.6159         0.2926         0.5561        0.2831       60
          ridge           2    0.6798   0.5533  0.3625          0.6140         0.2921         0.5542   